# T04 · Add-on: key and swing detection
*Music Information Retrieval & Machine Learning for Audio · TU Berlin · WiSe 2026/27*

In [ ]:
# Standard library
import sys
from pathlib import Path

# Third-party
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display
import torch
from demucs.api import Separator
from IPython.display import Audio, display

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────
CORPUS_DIR = Path("data") / "your-genre-name"   # ← name of your corpus folder
# ─────────────────────────────────────────────────────────────────────────
SESSION = "T05"
PREVIEW_DIR = CORPUS_DIR / "previews"
STEM_DIR = CORPUS_DIR / "stems"
STEM_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("output") / SESSION      # results of this session
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SR = xxx                                    # choose the sample rate for the outputs in this notebook
TRACK_ID = "xxx"                             # choose a track ID as example to work in within this notebook
TRACK = CORPUS_DIR / "previews" / f"{TRACK_ID}.mp3"

assert CORPUS_DIR.is_dir(), f"Not found: {CORPUS_DIR.resolve()} → check CORPUS_DIR"
CHART_CSV = next(CORPUS_DIR.glob("*.csv"))
print(CHART_CSV)

---
## 1 · Stem separation
**Approaches:**

| Approach | Package | Separates vocals? | Note |
|---|---|---|---|
| Harmonic/percussive (HPSS) | `librosa.effects.hpss` | no | fast; only tonal vs. percussive |
| Repetition-based (REPET-SIM) | `librosa.decompose.nn_filter` | partly | looped vocal chops end up in the "background" |
| NMF | `librosa.decompose.decompose` | not by itself | components must be assigned to sources by hand |
| Spleeter (Deezer) | `spleeter` | yes | TensorFlow, no longer runs with current Python versions |
| Open-Unmix | `openunmix` | yes | lighter, audibly worse |
| **Demucs (htdemucs)** | `demucs` | yes | state of the art; 4 stems: drums, bass, other, vocals |

In [ ]:
separator = Separator(model="htdemucs")      # loads the model (downloads the weights on first use)

# The model separates into 4 sources ("stems") and works internally at 44.1 kHz
print("Sources:", separator.model.sources, "| model sample rate:", separator.samplerate)

In [ ]:
STEMS = ["drums", "bass", "other", "vocals"]

def separate(mp3_path, out_dir=STEM_DIR):
    # Separate one preview with Demucs, save each stem as mono FLAC (SR) to out_dir/<track_id>/<stem>.flac
    # output convention
    track_dir = out_dir / mp3_path.stem
    paths = {s: track_dir / f"{s}.flac" for s in STEMS}

    # check if stems already exist
    if all(p.exists() for p in paths.values()):          # already done → skip
        return paths

    # the model expects a stereo file with 44.1kHz SR
    y, _ = librosa.load(mp3_path, sr=separator.samplerate, mono=False)
    y = np.atleast_2d(y)                                    # make sure shape ist as expected by the model
    # duplicate channels if you passed a mono file
    if y.shape[0] == 1:
        y = np.repeat(y, 2, axis=0)                        
    # convert your numpy array into PyTorch tensor
    wav = torch.from_numpy(y).float()

    # separation process
    _, stems = separator.separate_tensor(wav, separator.samplerate)
    # save each stem
    track_dir.mkdir(parents=True, exist_ok=True)
    for s, path in paths.items():
        stem = stems[s].mean(0).numpy()                     # convert stereo to mono or comment out, if you prefer to store the stereo file
        stem = librosa.resample(stem, orig_sr=separator.samplerate, target_sr=SR)  # 44.1 -> target SR
        sf.write(path, np.clip(stem, -1, 1), SR)            # FLAC is 16 bit -> clip values beyond ±1 to avoid clicks
    return paths

# return example and show result files
stem_paths = separate(TRACK)
for s, p in stem_paths.items():
    print(f"{s:<7} {p}  ({p.stat().st_size / 1e6:.1f} MB)")

**Listen** to every stem. Typical artefacts: hi-hats in the vocal stem (sound like sibilants),
synth leads detected as vocals, reverb tails smeared across stems.
`normalize=False` keeps the real level, so quiet residue *stays* quiet.

In [ ]:
# Play each stem one after the other
for s, p in stem_paths.items():
    y, _ = sf.read(p, dtype="float32")
    print(s)
    display(Audio(y, rate=SR, normalize=False))

---
## 2 · Key detection on the vocal stem
**Examples:**

| Approach | Package
|---|---
| **Template matching (Krumhansl-Schmuckler)**: chroma → mean profile → correlation with 24 key profiles | `librosa` (chroma) + `numpy`
| Other key profiles, e.g. Temperley, or **EDMA** (Faraldo et al. 2016, derived from electronic dance music) | `essentia` (`KeyExtractor`)
| CNN key classifier (Korzeniowski & Widmer 2018) | `madmom`

→ We do **Krumhansl-Schmuckler** by hand:
1. **Chromagram** of the vocal stem (`chroma_cqt`): energy per pitch class (C, C♯, …, B) per frame.
   `chroma_cqt` estimates the tuning deviation itself (lecture: tuning frequency).
2. **Ignore near-silent frames:** each chroma frame is normalised, so a silent frame (only separation residue)
   would count as much as a sung one.
3. **Mean profile** over the remaining frames = the tonal "fingerprint" of the vocals.
4. **Correlate** it with the pre-defined major and minor profile, rotated to all 12 tonics -> 24 correlations -> the highest correlation should be the key.

In [ ]:
# Load the vocal stem saved in block 1
vocals, _ = sf.read(stem_paths["vocals"], dtype="float32")

# Define your desired hop length (in samples) -> overlap between frames
HOP = 512                                   # librosa default (23 ms) – fine for pitch content

# chroma_cqt: for each frame, how much energy falls on each of the 12 pitch classes?
# (C, C#, D, …, B), regardless of the octave.
# Result: a table with 12 rows (pitch classes) × n columns (frames).
chroma = librosa.feature.chroma_cqt(y=vocals, sr=SR, hop_length=HOP)    # shape (12, n_frames)

# Problem: chroma normalises EVERY frame to the same maximum. A silent frame
# (only quiet separation residue) would count as much as a sung note.
SILENCE_DB = -40                            # <- your choice: how much dB below the loudest frame should be ignored?

# Loudness per frame: RMS relative to the loudest frame (in dB)
# [0] takes the single row out of the (1, n) result
rms_db = librosa.amplitude_to_db(librosa.feature.rms(y=vocals, hop_length=HOP)[0], ref=np.max)

n = min(chroma.shape[1], len(rms_db))  # both should have the same number of frames
active = rms_db[:n] > SILENCE_DB        # True/False per frame: loud enough?
print(f"{active.sum()} of {n} frames used ({100 * active.mean():.0f} %)")

# Keep only the active frames ([:, active]) and average over time (axis=1)
# -> 12 numbers: how present each pitch class is in the vocals overall
profile = chroma[:, :n][:, active].mean(axis=1)                      

In [ ]:
# Krumhansl-Schmuckler key profiles (Krumhansl & Kessler 1982)
# how well each of the 12 pitch classes "fits" into a key (measured in listening experiments)
# Index 0 = tonic, index 7 = fifth, etc. Here written for C major / C minor.
# Be aware that there are a lot more other scales (e.g. dorian, phrygian etc.) that might be applied here!
KS_MAJOR = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
KS_MINOR = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
PITCHES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

def key_correlations(profile):
    # Compare our 12-number profile with all 24 keys (12 tonics × major/minor).
    # Returns: key names and one correlation value r per key.
    keys, r = [], []
    for mode, template in (("major", KS_MAJOR), ("minor", KS_MINOR)):
        for tonic in range(12):
            keys.append(f"{PITCHES[tonic]} {mode}")
            # np.roll shifts the C template so that its tonic lands on another pitch class,
            # e.g. tonic=9 → A major / A minor.
            # np.corrcoef = Pearson correlation: +1 = same shape, 0 = unrelated, −1 = opposite
            r.append(np.corrcoef(profile, np.roll(template, tonic))[0, 1]) 
    return np.array(keys), np.array(r)

keys, r = key_correlations(profile)

# Sort keys from best to worst match (argsort sorts ascending → [::-1] reverses)
order = np.argsort(r)[::-1]
for k in order[:3]:             # print the 3 best candidates
    print(f"{keys[k]:<9} r = {r[k]:.2f}")
best_key = keys[order[0]]       # the winner = detected key

The **second closest match** matters as much as the winner: a low maximum *or* a small gap to the second key means the decision is weak. (Major and its relative minor, e.g. C major ↔ A minor, share all notes and often end up close together.)

-> Be aware: the main harmonics might not be within the vocal stem and/or the track/s are not on a major/minor scale!

In [ ]:
# Plot chromagram over time, mean profile as bar plot and correlation with all 24 keys (highest r = red)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), gridspec_kw={"width_ratios": [3, 1, 2]})
librosa.display.specshow(chroma[:, :n] * active, y_axis="chroma", x_axis="time",
                         sr=SR, hop_length=HOP, ax=axes[0])
axes[0].set(title=f"Chromagram vocal stem (silent frames set to 0) – {TRACK_ID}")
axes[1].barh(PITCHES, profile)
axes[1].set(title="Mean profile", xlabel="mean chroma")
axes[2].bar(range(24), r, color=["C0"] * 12 + ["C1"] * 12)
axes[2].bar(order[0], r[order[0]], color="C3")
axes[2].set_xticks(range(24), [k.replace(" major", "").replace(" minor", "m") for k in keys], rotation=90, fontsize=7)
axes[2].set(title=f"Correlation with 24 keys → {best_key}", ylabel="r")
fig.tight_layout()
plt.show()


---
## 3 · Ideas for "making the key result audible" 
There is no single right way to do this. First you should decide **what "not detected" means** for you
(e.g. low maximum r, small gap to the runner-up, too few active vocal frames, no vocals at all), then choose listening examples that make *that* reason audible.

Some possible approaches (combine or invent your own):

| Idea | What you hear | Building blocks |
|---|---|---|
| **Reference drone** | Vocal stem + a quiet chord (tonic triad) of the detected key and of the runner-up. Does the voice fit one, both, or neither? | sine tones via `librosa.midi_to_hz`, `librosa.note_to_midi`; add to the stem |
| **Only the frames that counted** | Concatenate just the frames above the silence threshold: is it really singing, or residue, shouts, hi-hats? | the `active` mask from block 2, frame ↔ sample conversion (`librosa.frames_to_samples`) |
| **What the algorithm "hears"** | Resynthesise the chromagram (e.g. one sine tone per strongest pitch class per frame) and compare it with the voice | `chroma`, `np.argmax`, sine synthesis |
| **Melody instead of chroma** | Pitch-track the vocal stem and play the extracted melody as a sine: is there a pitched melody at all (vs. rap, chops, spoken samples)? | `librosa.pyin` → `f0`, `voiced_flag` (lecture: melody extraction) |
| **Compare inputs** | Same track as vocal stem, full mix and mix without drums (or `librosa.effects.harmonic`): is the separation the problem, or the music itself? | stems from block 1, HPSS |
| **Key over time** | Key per window (e.g. 8–16 s) instead of one global key; play the windows that disagree: key change between breakdown and drop? | chroma averaged per window, same `key_correlations` |
| **Curated examples by cause** | One short labelled excerpt per failure type, e.g. "no sung melody", "modal / not major-minor", "synth lead in vocal stem", "separation residue" | excerpts + a small label table |

Remember for the homework: embed at least 1 examples with `Audio(..., embed=True)` and say in one sentence what the listener should notice in each one.

---
## 4 · Swing: inter-onset histogram of the drum stem
**Swing**: in a pair of eighth notes, the first is lengthened and the second shortened.
Friberg & Sundström (2002) measure this as the *swing ratio* of the long to the short note
(1 : 1 = straight, 2 : 1 = triplet feel).
Here, as in the task and on drum machines, we express it as the **share of the first note in the pair**:
swing (%) = long / (long + short) → **50 %** = straight, **66.7 %** = triplet swing.

**Approaches:**

| Approach | Function | + | − |
|---|---|---|---|
| **Inter-onset intervals (IOI) histogram** – hint on the task sheet | `onset_detect` + `np.diff` | no beat grid needed for the histogram; directly readable | needs consecutive onset pairs; levels can mix |
| Position of onsets within the beat | `beat_track` + onsets | separates 8th/16th level | depends on the beat grid |
| Tempogram ratio (energy at 8th / triplet ratios) | `librosa.feature.tempogram_ratio` | no onset picking | a measure of "ternary feel", not a swing % |
| RNN beat/downbeat tracking | `madmom`, `BeatNet` | state of the art | installation |

-> In this case we use the **IOI histogram** on the drum stem, in units of beats.

**Steps:**
1. **Onsets** of the drum stem with a fine time grid: Choose a `hop_length` that is not too coarse!
2. **Beats** (`beat_track`) → beat period → IOIs expressed as fractions of a beat (tempo-independent)
3. **Listen** to the onsets as clicks: wrong onsets = wrong swing.
4. **Histogram** of the IOIs: straight 8ths give one peak at 0.5 beat; swung 8ths split it into two peaks at *s* and 1 − *s*.

In [ ]:
# Load the drum stem saved in block 1
drums, _ = sf.read(stem_paths["drums"], dtype="float32")

# Swing differences are only ~20–40 ms, so you need a finer grid (e.g. 256, 128, 64...)
HOP_ON = 256           

# 1. Onset strength (novelty curve): how much "new" energy appears in each frame
env = librosa.onset.onset_strength(y=drums, sr=SR, hop_length=HOP_ON)

# 2. Pick the peak on that curve -> onset times in seconds
onsets = librosa.onset.onset_detect(onset_envelope=env, sr=SR, hop_length=HOP_ON, units="time")

# 3. Beat tracking (dynamic programming, Ellis 2007) on the same curve
#    -> estimated tempo + beat times in seconds
tempo, beats = librosa.beat.beat_track(onset_envelope=env, sr=SR, hop_length=HOP_ON, units="time")
tempo = float(np.atleast_1d(tempo)[0])   

# Length of one beat in seconds: median (robust against single wring beats) of the gaps between consecutive beats
beat_period = np.median(np.diff(beats))     

print(f"{len(onsets)} onsets | tempo ≈ {tempo:.1f} BPM | beat period {1000 * beat_period:.0f} ms")

Check by ear before measuring anything. Half or double tempo (e.g. 62 instead of 124 BPM) shifts all values by a factor of 2.

In [ ]:
# librosa.clicks creates a signal with a short click at each given time
# High click (3 kHz) = detected onset, low click (1 kHz) = detected beat
# length=len(drums) -> same length as the drum stem, so the two can be added
clicks_on = librosa.clicks(times=onsets, sr=SR, length=len(drums), click_freq=3000)
clicks_beat = librosa.clicks(times=beats, sr=SR, length=len(drums), click_freq=1000, click_duration=0.05)

# Mix drums + clicks (clicks at 30 % level)
# Listen: do the clicks sit exactly on the hits?
print("drum stem solo")
display(Audio(drums, rate=SR))
print("drum stem + onset clicks (high)")
display(Audio(drums + 0.3 * clicks_on, rate=SR))
print("drum stem + beat clicks (low)")
display(Audio(drums + 0.3 * clicks_beat, rate=SR))

In [ ]:

# Inter-onset intervals (IOIs): time between each onset and the next one
# np.diff([0.0, 0.29, 0.48]) -> [0.29, 0.19]
# Dividing by beat_period expresses them in beats instead of seconds which makes it independent of tempo:
# 0.5 = an 8th note, 0.25 = a 16th note, 1.0 = a quarter note (one beat)
ioi = np.diff(onsets) / beat_period 

fig, ax = plt.subplots()
ax.hist(ioi, bins=np.arange(0, 1.26, 0.01))
for x, label in [(0.25, "16th"), (1/3, ""), (0.5, "8th"), (2/3, ""), (1, "beat")]:
    ax.axvline(x, color="0.5", ls=":" if label else "--")
    if label:
        ax.text(x, ax.get_ylim()[1] * 0.95, label, ha="center", fontsize=8)
ax.set(xlabel="inter-onset interval (beats)   [dashed: triplet positions 1/3, 2/3]", ylabel="count",
       title=f"Inter-onset histogram drum stem – {TRACK_ID} ({tempo:.0f} BPM)")
plt.show()

**From histogram to a number.** Following the definition (paired eighths): take each onset that falls **on a beat**,
then the next two inter-onset intervals *a* and *b*. If together they span one beat, they form an 8th-note pair,
and the swing ratio is *a* / (*a* + *b*).
Anchoring on the beat matters: otherwise the pairs (long, short) and (short, long) both count, and everything averages to 50 %.

`pair_len` sets the level: `1.0` = 8th-note pairs (one beat), `0.5` = 16th-note pairs (half a beat).

In [ ]:
def swing_ratios(onsets, beats, pair_len=1.0, tol=0.1):
    # Find note pairs and compute their swing ratio a / (a + b):
    #   a = length of the 1st note, b = length of the 2nd note
    #   50 % = both equal (straight), 66.7 % = triplet swing
    # pair_len: length of one pair in beats -> 1.0 = two 8ths, 0.5 = two 16ths
    # tol: tolerance (10 %) for "starts on the grid" and "pair is complete"
    period = np.median(np.diff(beats))  # length (s)

    # For every onset: in which beat does it fall? (index of the last beat before it)
    # -1 means: the onset comes before the first detected beat
    i = np.searchsorted(beats, onsets, side="right") - 1         

    # Position of each onset within its beat: 0 = on the beat, 0.5 = halfway
    phase = (onsets - beats[np.clip(i, 0, None)]) / period      

    # Distance to the nearest grid point (in units of pair_len)
    # 8th level: grid = beats; 16th level: grid = beats + 8th positions
    grid_dist = np.abs(phase / pair_len - np.round(phase / pair_len))

    ratios = []
    for k in range(len(onsets) - 2):
        if i[k] < 0 or grid_dist[k] > tol:                         # pair must start on the grid
            continue
        a, b = np.diff(onsets[k:k + 3]) / period
        if abs((a + b) - pair_len) < tol * pair_len:               # both notes together = one pair
            ratios.append(a / (a + b))                             # -> otherwise: missing or extra onset
    return np.array(ratios)

# 8th-note level
ratios8 = swing_ratios(onsets, beats, pair_len=1.0)
if len(ratios8):
    print(f"8th-note pairs: n = {len(ratios8)}, median swing ratio = {100 * np.median(ratios8):.1f} %")
else:
    print("no 8th-note pairs found – e.g. continuous 16ths, or try pair_len=0.5")

# Distribution of the swing ratios of all pairs found (in %)
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.hist(100 * ratios8, bins=np.arange(40, 75, 1))
ax.axvline(50, color="k", ls=":", label="straight 50 %")
ax.axvline(200 / 3, color="k", ls="--", label="triplet 66.7 %")
ax.set(xlabel="swing ratio (%)", ylabel="pairs", title="8th-note pairs")
ax.legend(fontsize=7)
plt.show()